# Домашнее задание: AI Agents

## Инструкция
1. **Ячейка «Шаблон»** — запустите первой, не изменяйте.
2. **Ячейка с заданиями** — допишите код в местах `# ВАШ КОД ЗДЕСЬ`.
3. Запустите открытые примеры и убедитесь что все `OK`.
4. Сдайте ноутбук с сохранёнными output'ами.

In [12]:
%pip install -q langchain-openai langchain-core

In [13]:
%pip install --upgrade openai

In [14]:
from google.colab import userdata
import openai

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

YANDEX_FOLDER_ID= userdata.get('YANDEX_FOLDER_ID')

In [15]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          ШАБЛОН — НЕ ИЗМЕНЯТЬ ЭТУ ЯЧЕЙКУ                    ║
# ╚══════════════════════════════════════════════════════════════╝
# %pip install -q langchain-openai langchain-core

import os, json, copy
from typing import Any
from pathlib import Path
from dataclasses import dataclass, field

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool

MODEL_NAME = f"gpt://{YANDEX_FOLDER_ID}/gpt-oss-20b/latest"
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", OPENAI_API_KEY)
llm = ChatOpenAI(model=MODEL_NAME, temperature=0, api_key=OPENAI_API_KEY, base_url="https://ai.api.cloud.yandex.net/v1")


def llm_chat(messages: list, tools: list | None = None) -> AIMessage:
    """
    Отправляет историю сообщений в LLM и возвращает ответ модели.

    Параметры:
      messages — список сообщений диалога. Каждое сообщение — LangChain-объект:
                   SystemMessage(content="...")   — инструкция для модели (роль агента)
                   HumanMessage(content="...")    — сообщение от пользователя
                   AIMessage(...)                 — предыдущий ответ модели
                   ToolMessage(content="...", tool_call_id="...") — результат инструмента

      tools   — список описаний инструментов (схема OpenAI function calling или LangChain tools).

    Возвращает AIMessage:
      msg.content    — текстовый ответ (str)
      msg.tool_calls — список вызовов инструментов:
                         "name" — название инструмента
                         "args" — аргументы (уже распарсенный dict)
                         "id"   — уникальный идентификатор вызова
    """
    if tools:
        return llm.bind_tools(tools).invoke(messages)
    return llm.invoke(messages)


# Каталог товаров
CATALOG = [
    {"id": "p1",  "name": "Sony WH-1000XM5",            "category": "headphones", "brand": "Sony",     "price": 349, "color": "black",    "rating": 4.8, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p2",  "name": "Sony WH-CH720N",              "category": "headphones", "brand": "Sony",     "price": 129, "color": "blue",     "rating": 4.4, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p3",  "name": "Bose QuietComfort Ultra",     "category": "headphones", "brand": "Bose",     "price": 379, "color": "white",    "rating": 4.7, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p4",  "name": "Apple AirPods Pro 2",         "category": "earbuds",    "brand": "Apple",    "price": 249, "color": "white",    "rating": 4.6, "tags": ["wireless", "noise-cancelling", "ios"]},
    {"id": "p5",  "name": "Anker Soundcore Liberty 4 NC","category": "earbuds",    "brand": "Anker",    "price": 99,  "color": "black",    "rating": 4.3, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p6",  "name": "Logitech MX Master 3S",       "category": "mouse",      "brand": "Logitech", "price": 109, "color": "graphite", "rating": 4.8, "tags": ["wireless", "productivity", "premium"]},
    {"id": "p7",  "name": "Logitech Pebble 2",           "category": "mouse",      "brand": "Logitech", "price": 34,  "color": "white",    "rating": 4.2, "tags": ["wireless", "budget", "portable"]},
    {"id": "p8",  "name": "Keychron K2",                 "category": "keyboard",   "brand": "Keychron", "price": 89,  "color": "black",    "rating": 4.5, "tags": ["wireless", "mechanical", "compact"]},
    {"id": "p9",  "name": "NuPhy Air75",                 "category": "keyboard",   "brand": "NuPhy",    "price": 139, "color": "gray",     "rating": 4.6, "tags": ["wireless", "mechanical", "low-profile"]},
    {"id": "p10", "name": "Amazon Kindle Paperwhite",    "category": "ereader",    "brand": "Amazon",   "price": 149, "color": "black",    "rating": 4.7, "tags": ["reading", "portable", "gift"]},
]


@dataclass
class ShopState:
    """Состояние сессии: корзина и результаты последнего поиска."""
    cart: list = field(default_factory=list)
    last_results: list = field(default_factory=list)


@dataclass
class ToolCallRecord:
    name: str
    args: dict
    result: Any = None


class ToolTracer:
    """Собирает все вызовы инструментов."""
    def __init__(self):
        self.calls: list[ToolCallRecord] = []

    def record(self, name: str, args: dict, result: Any = None) -> None:
        self.calls.append(ToolCallRecord(name=name, args=args, result=result))

    def called(self, name: str) -> bool:
        return any(c.name == name for c in self.calls)

    def get_calls(self, name: str) -> list:
        return [c for c in self.calls if c.name == name]

    def print_trace(self) -> None:
        print("=== Трейс вызовов ===")
        for i, c in enumerate(self.calls, 1):
            print(f"  {i}. {c.name}({json.dumps(c.args, ensure_ascii=False)[:80]})")
            if c.result is not None:
                print(f"     -> {json.dumps(c.result, ensure_ascii=False)[:100]}")
        print("=====================")


class ShopTools:
    """Логика магазина — поиск и добавление в корзину."""
    def __init__(self, catalog):
        self.catalog = catalog

    def search_products(self, query: str = "", category: str | None = None,
                        brand: str | None = None, max_price: float | None = None,
                        sort_by: str | None = None) -> list:
        results = []
        q_words = query.lower().split() if query else []
        for item in self.catalog:
            hay = f"{item['name']} {item['category']} {item['brand']} {' '.join(item['tags'])}".lower()
            if q_words and not all(w in hay for w in q_words): continue
            if category and item["category"] != category: continue
            if brand and item["brand"].lower() != brand.lower(): continue
            if max_price is not None and item["price"] > float(max_price): continue
            results.append(copy.deepcopy(item))
        if sort_by == "price_asc": results.sort(key=lambda x: x["price"])
        elif sort_by == "rating_desc": results.sort(key=lambda x: -x["rating"])
        return results

    def add_to_cart(self, state: ShopState, product_id: str, quantity: int = 1) -> dict:
        product = next((p for p in self.catalog if p["id"] == product_id), None)
        if not product:
            return {"ok": False, "error": f"Товар {product_id} не найден"}
        existing = next((r for r in state.cart if r["product_id"] == product_id), None)
        if existing:
            existing["quantity"] += quantity
        else:
            state.cart.append({"product_id": product_id, "name": product["name"],
                                "price": product["price"], "quantity": quantity})
        return {"ok": True, "cart_size": len(state.cart)}


@dataclass
class AgentContext:
    """Общий контекст, передаваемый между агентами в задании 3."""
    query: str
    max_price: float | None = None
    candidates: list[dict] = field(default_factory=list)
    pros: dict[str, str] = field(default_factory=dict)   # product_id → описание плюсов
    cons: dict[str, str] = field(default_factory=dict)   # product_id → описание минусов
    best: dict | None = None
    cart_result: dict | None = None


TOOLS = ShopTools(CATALOG)
print("Шаблон загружен.")
print(f"  Модель: {MODEL_NAME}")
print(f"  Каталог: {len(CATALOG)} товаров")
print(f"  Утилиты: AgentContext, ToolTracer, ShopTools, convert_to_openai_tool")
print(f"  LangChain: HumanMessage, SystemMessage, AIMessage, ToolMessage")


Шаблон загружен.
  Модель: gpt://b1gke6787j46pb0eijv4/gpt-oss-20b/latest
  Каталог: 10 товаров
  Утилиты: AgentContext, ToolTracer, ShopTools, convert_to_openai_tool
  LangChain: HumanMessage, SystemMessage, AIMessage, ToolMessage


In [16]:
# ╔══════════════════════════════════════════════════════════════╗
# ║               ВАШ КОД — ТРИ ЗАДАНИЯ                          ║
# ╚══════════════════════════════════════════════════════════════╝

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ЗАДАНИЕ 1. Tool-Calling Agent (ReAct-цикл)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 1.1. Определи SHOP_TOOLS_SCHEMA — описание инструментов для LLM.
#
# Ниже даны функции-заглушки с сигнатурами, но без описаний.
# LLM нужно объяснить, что делает каждый инструмент и что значат его параметры.
#
# Задача: добавь docstring (описание + Args) к каждой функции.
# Функция convert_to_openai_tool() из шаблона сгенерирует JSON-схему автоматически.
# Подробности про формат докстрингов — см. Google-style docstrings.

def search_products(
    query: str = "",
    category: str | None = None,
    brand: str | None = None,
    max_price: float | None = None,
    sort_by: str | None = None,
) -> list:
    # ВАШ КОД ЗДЕСЬ: добавь docstring с описанием инструмента и параметров
    """
    Ищет товар в каталоге

    Args:
        query (str): текстовый запрос по названию, бренду, категории, тегам, например wireless headphones
        category (str | None): категория товара, например {headphones, earbuds, mouse, keyboard, ereader}
        brand (str | None): название бренда компании, например {Sony, Apple, Amazon}
        max_price (float | None): максимальная цена товара
        sort_by (str | None): в каком формате вернется результат функции, например price_asc - по возрастанию цены, rating_desc - по убыванию рейтинга

    Returns:
        list: массив найденных товаров
    """
    ...

def add_to_cart(product_id: str, quantity: int = 1) -> dict:
    """
    Добавляет товар(ы) в корзину
    Args:
        product_id (str): id товара
        quantity (int): количество товаров, по умолчанию 1

    Returns:
        dict: {
                "ok" (bool):состяние корзины,
                "cart_size" (int): длины корзины
                }
    """
    ...

# ВАШ КОД ЗДЕСЬ: сгенерируй схему
SHOP_TOOLS_SCHEMA = [
    convert_to_openai_tool(search_products),
    convert_to_openai_tool(add_to_cart),
]


# 1.2. Реализуй run_shopping_agent — ReAct-агент магазина.
def run_shopping_agent(user_message: str, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> str:
    """
    ReAct-агент магазина. Получает сообщение пользователя и итеративно:
      1. Вызывает LLM с историей и схемой инструментов.
      2. Если LLM вернула tool_calls — выполняет каждый инструмент:
           search_products → сохраняет результат в state.last_results, записывает в tracer
           add_to_cart     → добавляет товар в state.cart, записывает в tracer
         Добавляет ToolMessage с результатом в историю и повторяет цикл.
      3. Если tool_calls пустой — возвращает текстовый ответ LLM.
    """
    systemessage = "Ты полезный ассистент магазина. Используй инструменты (tools), чтобы искать товары или добавлять их в корзину. Не выдумывай товары. Добавляй в корзину (cart) только найденные товары через их id. Используй сортировку для 'самый дешевый' или 'лучший по рейтингу'"
    messages=[]
    messages.append(SystemMessage(systemessage))
    messages.append(HumanMessage(user_message))
    llm_answer = llm_chat(messages=messages, tools=SHOP_TOOLS_SCHEMA)
    messages.append(llm_answer)
    k=0
    while llm_answer.tool_calls and k<5:
        for i, call in enumerate(llm_answer.tool_calls):
            tool_name = call["name"]
            args = call["args"]
            print(args)
            tool_call_id = call["id"]
            tool_res = None
            if tool_name=='search_products':
                tool_res=tools.search_products(**args)
                state.last_results = tool_res
                tracer.record('search_products', args, tool_res)

            elif tool_name == 'add_to_cart':
                tool_res = tools.add_to_cart(state, **args)
                tracer.record('add_to_cart', args, tool_res)

            messages.append(ToolMessage(content=json.dumps(tool_res, ensure_ascii=False), tool_call_id=tool_call_id))
        llm_answer = llm_chat(messages=messages, tools=SHOP_TOOLS_SCHEMA)
        messages.append(llm_answer)
        k+=1
    return llm_answer.content






In [17]:
# ─── Открытые примеры для задания 1 ───────────────────────────

# [1.A] Поиск с фильтром по цене
_s1a = ShopState(); _t1a = ToolTracer()
_r1a = run_shopping_agent("Найди беспроводные наушники до 150 долларов", _s1a, TOOLS, _t1a)
_t1a.print_trace()
assert _t1a.called("search_products"), "FAIL: search_products не вызван"
assert all(p["price"] <= 150 for p in _s1a.last_results)
print("OK 1.A")
print(_r1a)

{'query': 'wireless headphones', 'category': 'headphones', 'max_price': 150, 'sort_by': 'price_asc'}
=== Трейс вызовов ===
  1. search_products({"query": "wireless headphones", "category": "headphones", "max_price": 150, "so)
     -> [{"id": "p2", "name": "Sony WH-CH720N", "category": "headphones", "brand": "Sony", "price": 129, "co
OK 1.A
Вот подходящие беспроводные наушники до 150 $:

| ID | Наименование | Бренд | Цена | Рейтинг |
|----|--------------|-------|------|---------|
| **p2** | Sony WH‑CH720N | Sony | 129 $ | 4.4 |

Если хотите добавить их в корзину, дайте знать!


In [18]:
# [1.B] Поиск + добавление самого дешёвого
_s1b = ShopState(); _t1b = ToolTracer()
_r1b = run_shopping_agent(
    "Найди беспроводную мышь дешевле 120 долларов и добавь самую дешёвую в корзину",
    _s1b, TOOLS, _t1b
)
assert _t1b.called("search_products") and _t1b.called("add_to_cart")
assert len(_s1b.cart) == 1 and _s1b.cart[0]["product_id"] == "p7"
print("OK 1.B")
print(_r1b)

{'query': 'wireless mouse', 'max_price': 120, 'sort_by': 'price_asc'}
{'product_id': 'p7', 'quantity': 1}
OK 1.B
✅ Беспроводную мышь «Logitech Pebble 2» (цена $34) добавили в корзину. Если понадобится что‑то ещё — дайте знать!


In [19]:
# [1.C] Лучшая клавиатура
_s1c = ShopState(); _t1c = ToolTracer()
_r1c = run_shopping_agent(
    "Найди беспроводную клавиатуру с лучшим рейтингом и добавь в корзину",
    _s1c, TOOLS, _t1c
)
assert _t1c.called("search_products") and _t1c.called("add_to_cart")
added = next(p for p in CATALOG if p["id"] == _s1c.cart[0]["product_id"])
assert added["category"] == "keyboard"
print(f"OK 1.C: '{added['name']}' (рейтинг {added['rating']})")
print(_r1c)

{'query': 'wireless keyboard', 'sort_by': 'rating_desc'}
{'product_id': 'p9', 'quantity': 1}
OK 1.C: 'NuPhy Air75' (рейтинг 4.6)
✅ Клавиатура **NuPhy Air75** (лучший рейтинг 4.6) добавлена в корзину. Если понадобится что‑то ещё — дайте знать!


In [20]:

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ЗАДАНИЕ 2. Memory Agent
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PROFILE_PATH = Path("user_profile.json")
# Рекомендуемые поля профиля:
#   name       — имя пользователя
#   brand      — предпочитаемый бренд
#   max_price  — максимальная цена
#   color      — предпочитаемый цвет
#   category   — интересующая категория

def load_profile(path: Path = PROFILE_PATH) -> dict:
    """Загружает профиль из JSON. Возвращает {} если файл не существует."""
    if path.exists():
      with open(path, 'r', encoding='utf') as file:
        profile = json.load(file)
      return profile
    else:
      return {}


def save_profile(profile: dict, path: Path = PROFILE_PATH) -> None:
    """Сохраняет словарь profile в файл как JSON."""
    with open(path, 'w', encoding="utf-8") as file:
      json.dump(profile, file,  ensure_ascii=False, indent=2)

def update_profile(key: str, value: str) -> dict:
  """
  Обновляет профиль пользователя, уточняет его предпочтения. Добавляй через
  этот инстурмент сведения о пользователе, например его любимый бренд или его имя.
  обязательно сохраняй информацию о пользователе, когда что-то про него узнаешь

  Args:
    key (str): ключ поля, которое хотим обновлять, например name, brand, max_price, color, category
    value (str): значение поля, например name: иван, brand: sony и т.д.

  Returns:
    dict: предпочтения пользователя
  """


SHOP_TOOLS_SCHEMA_WITH_MEMORY = [
    # ВАШ КОД ЗДЕСЬ — SHOP_TOOLS_SCHEMA + инструмент update_profile
    # update_profile: принимает key (рекомендуемые: name | brand | max_price | color | category)
    #                 и value — сохраняет предпочтение в профиль пользователя
    convert_to_openai_tool(update_profile),
    convert_to_openai_tool(search_products),
    convert_to_openai_tool(add_to_cart),

]

def run_memory_agent(
    user_message: str,
    state: ShopState,
    tools: ShopTools,
    tracer: ToolTracer,
    history: list,
    profile_path: Path = PROFILE_PATH,
) -> tuple:
    """
    Memory-агент. Расширяет run_shopping_agent долгосрочной и краткосрочной памятью.

    Долгосрочная память (Long-term):
      - Загружает профиль из файла (load_profile) при каждом запуске
      - Передаёт профиль агенту через SystemMessage
      - Инструмент update_profile обновляет профиль на диске при первом упоминании предпочтений

    Краткосрочная память (Short-term):
      - history содержит полную историю сообщений предыдущих ходов (включая ToolMessages)
      - Это позволяет агенту в следующем ходу «видеть» результаты прошлых поисков
      - Добавляется к запросу перед вызовом LLM

    Возвращает (ответ: str, обновлённая_история: list).
    Подсказка: сохраняй в историю ВСЕ сообщения (HumanMessage, AIMessage, ToolMessage),
    чтобы агент в следующем ходу знал что было найдено.
    """
    # ВАШ КОД ЗДЕСЬ
    profile = load_profile(profile_path)
    systemessage = f"""Ты полезный ассистент магазина.
    Вот сохранённый профиль пользователя: {profile}
    Используй его при ответах.
    Используй инструменты (tools), чтобы искать товары или добавлять их в корзину.
    Не выдумывай товары.
    Добавляй в корзину (cart) только найденные товары через их id.
    Используй сортировку для 'самый дешевый' или 'лучший по рейтингу'"""
    messages=[]
    messages.append(SystemMessage(systemessage))
    new_history=copy.deepcopy(history)
    for message in history:
      messages.append(message)

    messages.append(HumanMessage(user_message))
    new_history.append(HumanMessage(user_message))

    llm_answer = llm_chat(messages=messages, tools=SHOP_TOOLS_SCHEMA_WITH_MEMORY)
    messages.append(llm_answer)
    new_history.append(llm_answer)

    k=0
    while llm_answer.tool_calls and k<5:
        for i, call in enumerate(llm_answer.tool_calls):
            tool_name = call["name"]
            args = call["args"]
            print(args)
            tool_call_id = call["id"]
            tool_res = None
            if tool_name=='search_products':
                tool_res=tools.search_products(**args)
                state.last_results = tool_res
                tracer.record('search_products', args, tool_res)

            elif tool_name == 'add_to_cart':
                tool_res = tools.add_to_cart(state, **args)
                tracer.record('add_to_cart', args, tool_res)

            elif tool_name == 'update_profile':
                profile[args['key']] = args['value']
                tool_res = {"ok": True, "profile": profile}
                save_profile(profile, profile_path)
                tracer.record('update_profile', args, tool_res)

            new_history.append(ToolMessage(content=json.dumps(tool_res, ensure_ascii=False), tool_call_id=tool_call_id))
            messages.append(ToolMessage(content=json.dumps(tool_res, ensure_ascii=False), tool_call_id=tool_call_id))

        llm_answer = llm_chat(messages=messages, tools=SHOP_TOOLS_SCHEMA_WITH_MEMORY)
        messages.append(llm_answer)
        new_history.append(llm_answer)
        k+=1
    return (llm_answer.content, new_history)



In [21]:
# ─── Открытые примеры для задания 2 ───────────────────────────

# [2.A] Сохранение предпочтений
_p2a = Path("_demo_profile_2a.json")
if _p2a.exists(): _p2a.unlink()
_s2a = ShopState(); _t2a = ToolTracer(); _h2a = []
_r2a, _h2a = run_memory_agent(
    "Меня зовут Анна, я предпочитаю Sony и трачу не более 200 долларов",
    _s2a, TOOLS, _t2a, _h2a, _p2a
)
_prof2a = load_profile(_p2a)
print(_r2a)
print(_prof2a)
print(_t2a.print_trace())
assert _t2a.called("update_profile") and _prof2a.get("brand") == "Sony"
print("OK 2.A")

# [2.B] Новая сессия использует профиль (history=[])
_p2b = Path("_demo_profile_2b.json")
save_profile({"name": "Борис", "brand": "Logitech", "max_price": "150"}, _p2b)
_s2b = ShopState(); _t2b = ToolTracer(); _h2b = []
_r2b, _ = run_memory_agent("Как меня зовут и какой мой бюджет?", _s2b, TOOLS, _t2b, _h2b, _p2b)
assert "Борис" in _r2b
print("OK 2.B")
print(_r2b)

# [2.C] Short-term memory — ход 2 помнит ход 1
_p2c = Path("_demo_profile_2c.json")
if _p2c.exists(): _p2c.unlink()
_s2c = ShopState(); _h2c = []
_, _h2c = run_memory_agent(
    "Найди беспроводные наушники до 150 долларов", _s2c, TOOLS, ToolTracer(), _h2c, _p2c
)
assert len(_h2c) >= 2
_t2c2 = ToolTracer()
_, _h2c = run_memory_agent(
    "Добавь первый из найденных в корзину", _s2c, TOOLS, _t2c2, _h2c, _p2c
)
assert _t2c2.called("add_to_cart") and len(_s2c.cart) == 1
print(f"OK 2.C: добавлен '{_s2c.cart[0]['name']}'")
print(_)

{'key': 'name', 'value': 'Анна'}
{'key': 'brand', 'value': 'Sony'}
{'brand': 'Sony', 'max_price': 200}
Найдено одно подходящее предложение:

| ID | Товар | Цена | Рейтинг |
|----|-------|------|---------|
| **p2** | Sony WH‑CH720N (беспроводные наушники) | 129 $ | 4.4 |

Это наушники Sony, цена ниже вашего лимита в 200 $ и они получили хороший рейтинг.  
Хотите добавить их в корзину?
{'name': 'Анна', 'brand': 'Sony'}
=== Трейс вызовов ===
  1. update_profile({"key": "name", "value": "Анна"})
     -> {"ok": true, "profile": {"name": "Анна", "brand": "Sony"}}
  2. update_profile({"key": "brand", "value": "Sony"})
     -> {"ok": true, "profile": {"name": "Анна", "brand": "Sony"}}
  3. search_products({"brand": "Sony", "max_price": 200})
     -> [{"id": "p2", "name": "Sony WH-CH720N", "category": "headphones", "brand": "Sony", "price": 129, "co
None
OK 2.A
OK 2.B
Ваше имя — Борис, а бюджет составляет 150 рублей.
{'query': 'wireless headphones', 'category': 'headphones', 'max_price': 150, '

In [44]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ЗАДАНИЕ 3. Multi-Agent System
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# Реализуй систему из четырёх агентов + оркестратор.
# Цель — подобрать лучший товар и честно рассказать его плюсы и минусы.
# Агенты работают по цепочке через общий объект AgentContext (определён в шаблоне).
#
# RetrieverAgent (LLM + инструменты)
#   Ищет до 5 релевантных товаров через search_products.
#   Заполняет ctx.candidates и ctx.max_price.
#   Важно: передавай только инструмент поиска (не add_to_cart).
#
# ProsAgent (LLM, без инструментов)
#   Для каждого товара из ctx.candidates находит 1-2 предложения плюсов.
#   Заполняет ctx.pros (dict: product_id → строка плюсов).
#   Записывает в tracer вызов "analyze_pros".
#
# ConsAgent (LLM, без инструментов)
#   Для каждого товара из ctx.candidates находит 1-2 предложения минусов.
#   Заполняет ctx.cons (dict: product_id → строка минусов).
#   Записывает в tracer вызов "analyze_cons".
#
# RankerAgent (без LLM — только логика)
#   Из ctx.candidates выбирает лучший товар:
#     - Фильтрует по ctx.max_price (если задан)
#     - Среди прошедших: наибольший рейтинг; при равном — наименьшая цена
#   Записывает в tracer вызов "rank_candidates". Заполняет ctx.best.
#
# CoordinatorAgent (оркестратор)
#   Запускает агентов по цепочке, ведёт список trace.
#   Ключи трейса: "delegate_retriever", "delegate_pros", "delegate_cons",
#                 "delegate_ranker", "delegate_cart".
#   CartAgent не нужен — если пользователь просит добавить в корзину,
#   CoordinatorAgent делает это сам через tools.add_to_cart после ранжирования.
#   Возвращает AgentResult с ответом, трейсом и контекстом.
#   Ответ должен включать: название товара, цену, рейтинг, плюсы и минусы.

SHOP_TOOLS_SCHEMA_RetrieverAgent = [
    convert_to_openai_tool(search_products),
]
SHOP_TOOLS_SCHEMA_CoordinatorAgent = [
    convert_to_openai_tool(add_to_cart),
]

@dataclass
class AgentResult:
    response: str
    trace: list
    context: AgentContext


class RetrieverAgent:
    def run(self, ctx: AgentContext, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> AgentContext:
        """Ищет товары через LLM+tools. Заполняет ctx.candidates и ctx.max_price."""

        systemessage = f"""Ты полезный ассистент магазина для поиска товаров.
        Используй инструменты (tools), чтобы искать товары.
        Не выдумывай товары.
        Используй сортировку для 'самый дешевый' или 'лучший по рейтингу'
        если есть ограничение цены — передай max_price
        верни до 5 товаров"""

        messages=[]
        messages.append(SystemMessage(systemessage))

        messages.append(HumanMessage(ctx.query))

        llm_answer = llm_chat(messages=messages, tools=SHOP_TOOLS_SCHEMA_RetrieverAgent)
        messages.append(llm_answer)

        k=0

        while llm_answer.tool_calls and k<5:
            for i, call in enumerate(llm_answer.tool_calls):
                tool_name = call["name"]
                args = call["args"]
                print(args)
                tool_call_id = call["id"]
                tool_res = None
                if tool_name=='search_products':
                    tool_res=tools.search_products(**args)
                    state.last_results = tool_res
                    ctx.candidates = tool_res[:5]
                    if 'max_price' in args.keys():
                      ctx.max_price=args['max_price']
                    tracer.record('search_products', args, tool_res)

                messages.append(ToolMessage(content=json.dumps(tool_res, ensure_ascii=False), tool_call_id=tool_call_id))

            k+=1

        return ctx




class ProsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Находит плюсы каждого товара через LLM. Заполняет ctx.pros."""
        if ctx.candidates:
          for candidate in (ctx.candidates):
            message = f"""Ты полезный ассистент магазина для выбора товаров.
            кратко опиши плюсы этого товара: {candidate}
            """
            llm_answer = llm_chat(messages=[SystemMessage(message)])
            ctx.pros[candidate['id']] = llm_answer.content
            tracer.record("analyze_pros", candidate, llm_answer)
        return ctx




class ConsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Находит минусы каждого товара через LLM. Заполняет ctx.cons."""
        # ВАШ КОД ЗДЕСЬ
        if ctx.candidates:
          for candidate in (ctx.candidates):
            message = f"""Ты полезный ассистент магазина для выбора товаров.
            кратко опиши минусы этого товара: {candidate}
            """
            llm_answer = llm_chat(messages=[SystemMessage(message)])
            ctx.cons[candidate['id']] = llm_answer.content
            tracer.record("analyze_pros", candidate, llm_answer)
        return ctx





class RankerAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Выбирает лучший товар из ctx.candidates с учётом ctx.max_price. Заполняет ctx.best."""
        # ВАШ КОД ЗДЕСЬ
        best=None
        best_price=99999
        best_rating=0
        for candidate in ctx.candidates:
          if (best_rating<=candidate['rating']):
            if (best_price>candidate['price']):
              if ctx.max_price and (ctx.max_price>=candidate['price']) or ctx.max_price==None:
                best = candidate
                best_rating=candidate['rating']
                best_price=candidate['price']
        tracer.record('rank_candidates', ctx.candidates, best)

        ctx.best = best
        return ctx




class CoordinatorAgent:
    def __init__(self):
        self.retriever = RetrieverAgent()
        self.pros_agent = ProsAgent()
        self.cons_agent = ConsAgent()
        self.ranker = RankerAgent()

    def run(self, user_message: str, state: ShopState, tools: ShopTools) -> AgentResult:
        """Оркестрирует агентов. Возвращает AgentResult с ответом, трейсом и контекстом."""
        ctx = AgentContext(query=user_message)
        tracer = ToolTracer()
        trace=[]
        ctx=self.retriever.run(ctx=ctx, state=state, tools = tools, tracer=tracer)
        trace.append('delegate_retriever')

        ctx=self.pros_agent.run(ctx=ctx, tracer=tracer)
        trace.append('delegate_pros')

        ctx=self.cons_agent.run(ctx=ctx, tracer=tracer)
        trace.append('delegate_cons')

        ctx=self.ranker.run(ctx=ctx, tracer=tracer)
        trace.append('delegate_ranker')

        systemessage = f"""Ты полезный ассистент магазина.
        Используй инструменты (tools), чтобы добавлять товар в корзину, если пользователь этого просит
        пользователь уже задал вопрос: {user_message}
        мы нашли для него несколько товаров: {ctx.candidates}
        их плюсы: {ctx.pros}
        и минусы: {ctx.cons}.
        Выбрали лучший: {ctx.best}
        Выдай пользователю лучший товар с его описанием и добавь его в корзину, если пользователь этого просит
        е выдумывай товары.
        """

        messages=[]
        messages.append(SystemMessage(systemessage))

        messages.append(HumanMessage(user_message))

        llm_answer = llm_chat(messages=messages, tools=SHOP_TOOLS_SCHEMA_CoordinatorAgent)
        messages.append(llm_answer)

        k=0

        while llm_answer.tool_calls and k<5:
            for i, call in enumerate(llm_answer.tool_calls):
                tool_name = call["name"]
                args = call["args"]
                print(args)
                tool_call_id = call["id"]
                tool_res = None
                if tool_name == 'add_to_cart':
                  tool_res = tools.add_to_cart(state, **args)
                  tracer.record('add_to_cart', args, tool_res)
                  trace.append('delegate_cart')

                messages.append(ToolMessage(content=json.dumps(tool_res, ensure_ascii=False), tool_call_id=tool_call_id))
            llm_answer = llm_chat(messages=messages, tools=SHOP_TOOLS_SCHEMA_WITH_MEMORY)
            messages.append(llm_answer)

            k+=1
        return AgentResult(llm_answer, trace, ctx)



In [45]:
# ─── Открытые примеры для задания 3 ───────────────────────────

# [3.A] Полный цикл: поиск → плюсы → минусы → ранжирование → корзина
_s3a = ShopState()
_res3a = CoordinatorAgent().run(
    "Найди лучшую беспроводную мышь до 120 долларов и добавь в корзину", _s3a, TOOLS
)
print(_res3a.context.pros)
print(_res3a.context.cons)
print(_res3a.response)
assert "delegate_retriever" in _res3a.trace
assert "delegate_pros" in _res3a.trace and "delegate_cons" in _res3a.trace
assert "delegate_ranker" in _res3a.trace and "delegate_cart" in _res3a.trace
assert len(_s3a.cart) == 1 and _s3a.cart[0]["product_id"] == "p6"
assert _res3a.context.best is not None and _res3a.context.best["id"] == "p6"
assert len(_res3a.context.pros) > 0 and len(_res3a.context.cons) > 0
print("OK 3.A")

# [3.B] Только поиск без добавления
_s3b = ShopState()
_res3b = CoordinatorAgent().run("Найди беспроводную клавиатуру", _s3b, TOOLS)
assert "delegate_retriever" in _res3b.trace
assert "delegate_pros" in _res3b.trace and "delegate_cons" in _res3b.trace
assert "delegate_ranker" in _res3b.trace
assert "delegate_cart" not in _res3b.trace and len(_s3b.cart) == 0
assert _res3b.context.best is not None
print("OK 3.B")


{'query': 'wireless mouse', 'category': 'mouse', 'max_price': 120, 'sort_by': 'rating_desc'}
{'query': 'wireless mouse', 'category': 'mouse', 'max_price': 120, 'sort_by': 'rating_desc'}
{'query': 'wireless mouse', 'category': 'mouse', 'max_price': 120, 'sort_by': 'rating_desc'}
{'query': 'wireless mouse', 'category': 'mouse', 'max_price': 120, 'sort_by': 'rating_desc'}
{'query': 'wireless mouse', 'category': 'mouse', 'max_price': 120, 'sort_by': 'rating_desc'}
{'product_id': 'p6', 'quantity': 1}
{'p6': '**Плюсы Logitech MX Master 3S**\n\n- **Беспроводной** – свободное перемещение без проводов.  \n- **Эргономичный дизайн** – удобен для длительной работы.  \n- **Высокая производительность** – чувствительность до 4000\u202fDPI, быстрый скролл.  \n- **Персонализируемые кнопки** – легко настроить горячие клавиши.  \n- **Долговечная батарея** – до 70\u202fдней работы на одной зарядке.  \n- **Премиальный материал** – прочный корпус, качественная сборка.  \n- **Высокая оценка** – 4.8/5, подтве

In [46]:
# [3.C] RankerAgent — тай-брейк по цене при равном рейтинге
_ctx3c = AgentContext(query="тест", candidates=[
    {"id": "x1", "name": "A", "price": 200, "rating": 4.8},
    {"id": "x2", "name": "B", "price": 150, "rating": 4.8},
    {"id": "x3", "name": "C", "price": 100, "rating": 4.5},
])
_tr3c = ToolTracer()
_ctx3c = RankerAgent().run(_ctx3c, _tr3c)
print(_ctx3c)
_tr3c.print_trace()
print(_ctx3c.best)
assert _ctx3c.best["id"] == "x2" and _tr3c.called("rank_candidates")
print("OK 3.C")


AgentContext(query='тест', max_price=None, candidates=[{'id': 'x1', 'name': 'A', 'price': 200, 'rating': 4.8}, {'id': 'x2', 'name': 'B', 'price': 150, 'rating': 4.8}, {'id': 'x3', 'name': 'C', 'price': 100, 'rating': 4.5}], pros={}, cons={}, best={'id': 'x2', 'name': 'B', 'price': 150, 'rating': 4.8}, cart_result=None)
=== Трейс вызовов ===
  1. rank_candidates([{"id": "x1", "name": "A", "price": 200, "rating": 4.8}, {"id": "x2", "name": "B)
     -> {"id": "x2", "name": "B", "price": 150, "rating": 4.8}
{'id': 'x2', 'name': 'B', 'price': 150, 'rating': 4.8}
OK 3.C


In [47]:
# [3.D] RankerAgent учитывает ctx.max_price
_ctx3d = AgentContext(
    query="мышь до 120 долларов",
    max_price=120.0,
    candidates=[
        {"id": "expensive", "name": "Супер-мышь",  "price": 200, "rating": 4.9},
        {"id": "p6",        "name": "MX Master 3S", "price": 109, "rating": 4.8},
        {"id": "p7",        "name": "Pebble 2",      "price": 34,  "rating": 4.2},
    ],
)
_tr3d = ToolTracer()
_ctx3d = RankerAgent().run(_ctx3d, _tr3d)
print(_ctx3d)
print(_ctx3d.best)
assert _ctx3d.best is not None and _ctx3d.best["id"] == "p6"
print("OK 3.D: контекст передан корректно, max_price учитывается")

AgentContext(query='мышь до 120 долларов', max_price=120.0, candidates=[{'id': 'expensive', 'name': 'Супер-мышь', 'price': 200, 'rating': 4.9}, {'id': 'p6', 'name': 'MX Master 3S', 'price': 109, 'rating': 4.8}, {'id': 'p7', 'name': 'Pebble 2', 'price': 34, 'rating': 4.2}], pros={}, cons={}, best={'id': 'p6', 'name': 'MX Master 3S', 'price': 109, 'rating': 4.8}, cart_result=None)
{'id': 'p6', 'name': 'MX Master 3S', 'price': 109, 'rating': 4.8}
OK 3.D: контекст передан корректно, max_price учитывается
